# 直传模式介绍

从本节开始，我们将深入学习 HIXL 的 3 种传输模式，了解每种传输模式的适用场景以及实际性能，为实际业务中的链路/模式选择提供夯实的理论基础。本节将介绍 HIXL 的 3 种传输模式之一：直传模式。

本节学习大纲如下：

- 直传模式的含义
- 直传模式的优势和使用限制
- 直传模式的性能


## 1. 直传模式的含义

直传模式是 HIXL 默认开启的传输模式，利用底层的单边通信接口直接在已注册内存之间执行数据搬运，无需经过中间缓冲区。

下图展示了直传模式的数据传输流程：
<img src="./images/direct_transfer_path.png" alt="直传模式数据路径图" width="80%">

### 1.1 传输链路选择

直传模式根据服务器组网拓扑结构自动选择最优传输协议：

- A2 场景：机内默认适用 HCCS 传输，跨机默认使用 RoCE 传输；
- A3 场景：超节点内 机内和跨机 都默认使用 HCCS 传输。

可以通过设置环境变量 `HCCL_INTRA_ROCE_ENABLE=1` 强制使用 RoCE 协议传输，适用于需要统一网络管理或部分硬件链路不支持的场景。


## 2. 直传模式的优势和使用限制

直传模式是 HIXL 的默认模式，本节介绍直传默认的优势，以及一些使用场景上的限制，帮助你更好地了解何时可以使用直传模式。

### 2.1 直传模式的优势

**1. 零拷贝传输路径**

直传模式直接在源端和目标端内存之间执行数据搬运，消除了中间缓冲区的拷贝开销。数据只需经过一次物理传输即可到达目标位置，显著降低 CPU 和内存带宽消耗。

**2. 单边操作语义**

直传模式采用单边操作语义，发起方可以直接读写对端内存，无需对端进程主动参与。

**3. 批量操作能力**

单边接口支持批量提交传输请求，允许在一次 API 调用中完成多个不连续内存区域的传输，减少 API 调用开销，提升整体吞吐。

**4. 同步与异步双模式**

| 模式 | 接口 | 特点 | 适用场景 |
| --- | --- | --- | --- |
| 同步传输 | `TransferSync` | 阻塞等待完成 | 需要确认数据到达后再继续 |
| 异步传输 | `TransferAsync` | 立即返回，后台执行 | 计算与传输重叠执行 |

### 2.2 直传模式的使用限制

**1. 传输方向限制**

 - A2 场景：单机内使用 HCCS 传输时，只支持 D2rD。
 - A3 场景：使用 HCCS 传输时，不支持 Host 内存作为目标地址，即不支持 D2rH 和 H2rH。

**2. 内存申请限制**

使用 HCCS 传输时，底层驱动要求注册内存的首地址需要 2MB 对齐。通常有两种解决办法：

- 申请内存时指定 ALC_MEM_MALLOC_HUGE_ONLY 内存申请策略，强制申请大页内存。
- 申请普通页内存时，需要自己保证内存首地址 2M 对齐，对齐方法可参考如下示例：

假设用户申请的内存地址首地址为 addr，内存大小为 mem_size 字节，对齐内存块为 ALIGNMENT_BLOCK = 2 * 1024 *1024 字节，则对齐后的内存首地址为：
``` python
aligned_addr = (addr + ALIGNMENT_BLOCK - 1) // ALIGNMENT_BLOCK * ALIGNMENT_BLOCK
# 同时，为了避免地址2M对齐后的内存越界，注册的内存大小应减去偏移的大小：
aligned_mem_size = mem_size - (aligned_addr - addr)
```

### 2.3 Host 内存注册大小限制

**HDK 版本限制**

当 HDK 版本低于 25.5 时，由于 RDMA 内存页管理方式的限制，注册 Host 内存存在容量上限（约 20GB），超出此限制的 Host 内存注册将失败。

**解决方案**：

- 升级 HDK 至 25.5 或更高版本（推荐）
- 使用中转 Buffer 模式传输未注册的 Host 内存数据（旧 HDK 版本兼容方案，不推荐）


## 3. 直传模型的性能

下表只展示 A2/A3 场景下直传模式的实测 WRITE 写带宽，详细实测性能请查看 [A2 benchmark数据](https://gitcode.com/cann/hixl/blob/master/benchmarks/A2_benchmark_performance.md) 和 [A3 benchmark数据](https://gitcode.com/cann/hixl/blob/master/benchmarks/A3_benchmark_performance.md)。

- A2 场景的性能：

| 链路类型 | 机内实测带宽(GB/s) | 跨机实测带宽(GB/s) |
| --- | --- | --- |
| HCCS D2rD | 19 | 23 |
| RoCE D2rD | 23 | 23 |
| HCCS H2rD | 不支持 | 23 |
| RoCE H2rD | 23 | 23 |
| HCCS D2rH | 不支持 | 19 |
| RoCE D2rH | 19 | 23 |
| HCCS H2rH | 不支持 | 19 |
| RoCE H2rH | 18 | 23 |

- A3 场景的性能：

| 链路类型 | 机内实测带宽(GB/s) | 跨机实测带宽(GB/s) |
| --- | --- | --- |
| HCCS D2rD | 113 | 113 |
| RoCE D2rD | 22 | 22 |
| HCCS H2rD | 32 | 18 |
| RoCE H2rD | 22 | 22 |
| HCCS D2rH | 不支持 | 不支持 |
| RoCE D2rH | 22 | 22 |
| HCCS H2rH | 不支持 | 不支持 |
| RoCE H2rH | 22 | 22 |

> 实际业务中需要根据业务流量特点选择合适的传输链路，盲目地选择高带宽链路，不仅会造成流量负载不均衡，严重时还会因流量冲突导致带宽抢占，从而造成业务性能波动和下降。


## 课后练习

本节介绍了直传模式的原理、优势和使用限制，请根据学习内容完成以下题目进行自测。

1. （判断题）直传模式是 HIXL 默认开启的传输模式，数据通过中间缓冲区进行中转后再到达目标内存。

2. （判断题）A3 场景下使用 HCCS 传输时，不支持 Host 内存作为目标地址。

3. （单选题）如何强制 HIXL 使用 RoCE 协议进行传输？
    A. 设置环境变量 HCCL_INTRA_ROCE_ENABLE=0
    B. 设置环境变量 HCCL_INTRA_ROCE_ENABLE=1
    C. 设置环境变量 HCCS_ENABLE=0
    D. 设置环境变量 ROCE_ENABLE=1

4. （单选题）在 A2 场景下，关于直传模式的传输协议选择，以下说法正确的是？
    A. 机内和跨机都默认使用 RoCE 传输
    B. 机内默认使用 HCCS 传输，跨机默认使用 RoCE 传输
    C. 机内默认使用 RoCE 传输，跨机默认使用 HCCS 传输
    D. 机内和跨机都默认使用 HCCS 传输

5. （多选题）直传模式具有哪些优势？
    A. 零拷贝传输路径，消除中间缓冲区拷贝开销
    B. 单边操作语义，发起方可直接读写对端内存无需对端参与
    C. 批量操作能力，单次提交多个传输请求减少 API 调用开销
    D. 支持同步与异步双模式，适应不同业务场景

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/03.02_answer.txt